# Workshop — Delta Lake ✅ SOLUTION

> ⚠️ **This is the reference solution. Use only after attempting the workshop on your own.**

## Contents

| Task | Topic |
|------|-------|
| 01 | Create Delta table |
| 02 | Schema Enforcement |
| 03 | Schema Evolution |
| 04 | Constraints |
| 05 | INSERT |
| 06 | UPDATE |
| 07 | DELETE |
| 08 | MERGE INTO (upsert) |
| 09 | Version History |
| 10 | Time Travel — by version |
| 11 | Time Travel — by timestamp |
| 12 | RESTORE — Disaster Recovery |

In [0]:
%run ../setup/00_setup

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    DoubleType, DateType, TimestampType, LongType
)
from datetime import datetime, timedelta

TABLE = f"{CATALOG}.{BRONZE_SCHEMA}.lab_delta_cust_ws"

# Drop any leftover from a previous run
spark.sql(f"DROP TABLE IF EXISTS {TABLE}")

print(f"Working table : {TABLE}")
print(f"Dataset path  : {DATASET_PATH}")

---
## Task 01 — Create the Delta table

📖 [Delta Lake quickstart](https://docs.databricks.com/delta/quick-start.html)

In [0]:
customers_df = (
    spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(f"{DATASET_PATH}/customers/customers.csv")
)

customers_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(TABLE)

print(f"Source rows : {customers_df.count():,}")
print(f"Table rows  : {spark.table(TABLE).count():,}")
display(spark.table(TABLE).limit(5))

In [0]:
# ── CHECK 01 ──────────────────────────────────────────────────────
assert 'customers_df' in dir(), "customers_df not defined"
_tbl = spark.table(TABLE)
assert _tbl.count() > 0, "Delta table is empty"
assert _tbl.count() == customers_df.count(), \
    f"Row mismatch: source={customers_df.count()}, table={_tbl.count()}"
assert 'customer_id' in _tbl.columns, "Missing column: customer_id"
_fmt = spark.sql(f"DESCRIBE DETAIL {TABLE}").collect()[0]['format']
assert _fmt == 'delta', f"Table format must be 'delta', got '{_fmt}'"
print(f"✓ Task 01 passed — {_tbl.count():,} rows, format: delta")

---
## Task 02 — Schema Enforcement

📖 [Delta schema validation](https://docs.databricks.com/delta/schema-validation.html)

In [0]:
rows_before_bad = spark.table(TABLE).count()
enforcement_error = None

bad_df = spark.createDataFrame(
    [("BAD001", "Test", "Unknown", "test@bad.com", "Nowhere", "happy")],
    ["customer_id", "first_name", "last_name", "email", "country", "mood"]
)

try:
    bad_df.write.format("delta").mode("append").saveAsTable(TABLE)
except Exception as e:
    enforcement_error = str(e)

print(f"Caught error : {'YES' if enforcement_error else 'NO'}")
print(f"Rows before  : {rows_before_bad:,}  |  Rows now: {spark.table(TABLE).count():,}")

In [0]:
# ── CHECK 02 ──────────────────────────────────────────────────────
assert enforcement_error is not None, \
    "enforcement_error is None — the write should have raised an exception"
assert isinstance(enforcement_error, str) and len(enforcement_error) > 0, \
    "enforcement_error must be a non-empty string"
_rows_now = spark.table(TABLE).count()
assert _rows_now == rows_before_bad, \
    f"Row count changed! Schema enforcement failed — expected {rows_before_bad}, got {_rows_now}"
print(f"✓ Task 02 passed — Delta rejected bad schema, table unchanged ({_rows_now:,} rows)")

---
## Task 03 — Schema Evolution

📖 [ALTER TABLE — add columns](https://docs.databricks.com/sql/language-manual/sql-ref-syntax-ddl-alter-table.html) · [Delta schema evolution](https://docs.databricks.com/delta/update-schema.html)

In [0]:
spark.sql(f"ALTER TABLE {TABLE} ADD COLUMN loyalty_tier STRING")

display(spark.table(TABLE).select("customer_id", "loyalty_tier").limit(5))

In [0]:
# ── CHECK 03 ──────────────────────────────────────────────────────
_tbl = spark.table(TABLE)
assert 'loyalty_tier' in _tbl.columns, "Column 'loyalty_tier' was not added"
_fdict = {f.name: f.dataType for f in _tbl.schema.fields}
assert isinstance(_fdict['loyalty_tier'], StringType), \
    f"loyalty_tier must be StringType, got {_fdict['loyalty_tier']}"
_null_count = _tbl.filter(F.col("loyalty_tier").isNull()).count()
assert _null_count == _tbl.count(), \
    f"All existing rows should have NULL loyalty_tier — {_null_count}/{_tbl.count()} are NULL"
print(f"✓ Task 03 passed — loyalty_tier (STRING) added, all {_null_count:,} existing rows = NULL")

---
## Task 04 — Constraints

📖 [Delta table constraints](https://docs.databricks.com/tables/constraints.html)

In [0]:
constraint_error = None

# Step 1 — add the CHECK constraint
spark.sql(f"""
    ALTER TABLE {TABLE}
    ADD CONSTRAINT valid_country CHECK (country IS NOT NULL)
""")

# Step 2 — inspect active constraints
spark.sql(f"SHOW TBLPROPERTIES {TABLE}") \
    .filter(F.col("key").startswith("delta.constraints")) \
    .show(truncate=False)

# Step 3 — try to INSERT a violating row; catch and store the error
try:
    spark.sql(f"""
        INSERT INTO {TABLE} (customer_id, first_name, last_name, email, country)
        VALUES ('BAD_C', 'NullCountry', 'Test', 'nc@test.com', NULL)
    """)
except Exception as e:
    constraint_error = str(e)

print(f"Constraint violated : {'YES' if constraint_error else 'NO (constraint may not be set)'}")

In [0]:
# ── CHECK 04 ──────────────────────────────────────────────────────
_props = {r['key']: r['value']
          for r in spark.sql(f"SHOW TBLPROPERTIES {TABLE}").collect()}

# Constraint must be registered under the expected name
assert 'delta.constraints.valid_country' in _props, \
    "Constraint 'valid_country' not found — expected key 'delta.constraints.valid_country' in SHOW TBLPROPERTIES"

# Constraint expression must reference the correct column
_expr = _props['delta.constraints.valid_country'].lower()
assert 'country' in _expr and 'null' in _expr, \
    f"Constraint expression looks wrong: '{_props['delta.constraints.valid_country']}' — should check that country IS NOT NULL"

# Violation must have been caught
assert constraint_error is not None, \
    "constraint_error is None — the violating INSERT should have raised an exception"

# BAD_C must not be in the table
_bad = spark.table(TABLE).filter(F.col("customer_id") == "BAD_C").count()
assert _bad == 0, "Row with NULL country was inserted — constraint did not fire"

print(f"✓ Task 04 passed — constraint 'valid_country' active: {_props['delta.constraints.valid_country']}")
print(f"  Violating INSERT correctly rejected.")

---
## Task 05 — INSERT

📖 [INSERT INTO — Delta DML](https://docs.databricks.com/sql/language-manual/sql-ref-syntax-dml-insert-into.html)

In [0]:
count_before_insert = spark.table(TABLE).count()

spark.sql(f"""
    INSERT INTO {TABLE} (customer_id, first_name, last_name, email, country)
    VALUES
        ('TEST001', 'Alice',   'Test', 'alice@test.com',   'TestLand'),
        ('TEST002', 'Bob',     'Test', 'bob@test.com',     'TestLand'),
        ('TEST003', 'Charlie', 'Test', 'charlie@test.com', 'TestLand')
""")

count_after_insert = spark.table(TABLE).count()
print(f"Before: {count_before_insert:,} | After: {count_after_insert:,} | Inserted: {count_after_insert - count_before_insert}")

In [0]:
# ── CHECK 05 ──────────────────────────────────────────────────────
_after = spark.table(TABLE).count()
assert _after == count_before_insert + 3, \
    f"Expected {count_before_insert + 3} rows, got {_after}"
_ids = {r['customer_id'] for r in spark.table(TABLE).select("customer_id").collect()}
for _tid in ['TEST001', 'TEST002', 'TEST003']:
    assert _tid in _ids, f"Inserted customer not found: {_tid}"
_hist = [r['operation'] for r in spark.sql(f"DESCRIBE HISTORY {TABLE}").limit(3).collect()]
assert 'WRITE' in _hist, "INSERT must create a new WRITE version in Delta history"
print(f"✓ Task 05 passed — {_after:,} rows total, INSERT logged in Delta history")

---
## Task 06 — UPDATE

📖 [UPDATE — Delta DML](https://docs.databricks.com/sql/language-manual/delta-update.html)

In [0]:
spark.sql(f"""
    UPDATE {TABLE}
    SET country = 'Poland', loyalty_tier = 'Gold'
    WHERE customer_id = 'TEST001'
""")

display(spark.table(TABLE).filter(F.col("customer_id") == "TEST001"))

In [0]:
# ── CHECK 06 ──────────────────────────────────────────────────────
_row = spark.table(TABLE).filter(F.col("customer_id") == "TEST001").collect()
assert len(_row) == 1, "TEST001 must still exist after UPDATE"
assert _row[0]['country']      == 'Poland', f"country should be 'Poland', got '{_row[0]['country']}'"
assert _row[0]['loyalty_tier'] == 'Gold',   f"loyalty_tier should be 'Gold', got '{_row[0]['loyalty_tier']}'"
_hist = [r['operation'] for r in spark.sql(f"DESCRIBE HISTORY {TABLE}").limit(5).collect()]
assert 'UPDATE' in _hist, "UPDATE must appear in Delta history"
print("✓ Task 06 passed — TEST001 updated to Poland/Gold, new version in history")

---
## Task 07 — DELETE

📖 [DELETE FROM — Delta DML](https://docs.databricks.com/sql/language-manual/delta-delete-from.html)

In [0]:
count_before_delete = spark.table(TABLE).count()

spark.sql(f"""
    DELETE FROM {TABLE}
    WHERE customer_id IN ('TEST002', 'TEST003')
""")

count_after_delete = spark.table(TABLE).count()
print(f"Before: {count_before_delete:,} | After: {count_after_delete:,} | Deleted: {count_before_delete - count_after_delete}")

In [0]:
# ── CHECK 07 ──────────────────────────────────────────────────────
_remaining = spark.table(TABLE).count()
assert _remaining == count_before_delete - 2, \
    f"Expected {count_before_delete - 2} rows, got {_remaining}"
_ids = {r['customer_id'] for r in spark.table(TABLE).select("customer_id").collect()}
assert 'TEST002' not in _ids, "TEST002 was not deleted"
assert 'TEST003' not in _ids, "TEST003 was not deleted"
assert 'TEST001' in _ids,     "TEST001 should still exist (was not in the DELETE)"
_hist = [r['operation'] for r in spark.sql(f"DESCRIBE HISTORY {TABLE}").limit(5).collect()]
assert 'DELETE' in _hist, "DELETE must appear in Delta history"
print(f"✓ Task 07 passed — {_remaining:,} rows remain, DELETE logged in history")

---
## Task 08 — MERGE INTO (upsert)

📖 [MERGE INTO — Delta DML](https://docs.databricks.com/sql/language-manual/delta-merge-into.html)

In [0]:
# Incoming batch — do not modify
updates_data = [
    ("TEST001",    "MergedAlice",  "Smith",  "merged@test.com",   "France",  "Platinum"),
    (customers_df.first()["customer_id"], "MergedProd", "X", "mp@test.com", "Spain", "Silver"),
    ("MERGE_NEW",  "NewViaMerge",  "Lab",    "new@merge.com",     "Germany", None),
]
updates_df = spark.createDataFrame(
    updates_data,
    ["customer_id", "first_name", "last_name", "email", "country", "loyalty_tier"]
)
updates_df.createOrReplaceTempView("updates_view")
display(updates_df)

In [0]:
count_before_merge = spark.table(TABLE).count()

spark.sql(f"""
    MERGE INTO {TABLE} AS target
    USING updates_view AS source
    ON target.customer_id = source.customer_id
    WHEN MATCHED THEN UPDATE SET
        target.first_name   = source.first_name,
        target.last_name    = source.last_name,
        target.email        = source.email,
        target.country      = source.country,
        target.loyalty_tier = source.loyalty_tier
    WHEN NOT MATCHED THEN INSERT
        (customer_id, first_name, last_name, email, country, loyalty_tier)
    VALUES
        (source.customer_id, source.first_name, source.last_name,
         source.email, source.country, source.loyalty_tier)
""")

count_after_merge = spark.table(TABLE).count()
print(f"Before: {count_before_merge:,} | After: {count_after_merge:,} | Net change: {count_after_merge - count_before_merge}")

In [0]:
# ── CHECK 08 ──────────────────────────────────────────────────────
_after_merge = spark.table(TABLE).count()
assert _after_merge == count_before_merge + 1, \
    f"MERGE should insert 1 new row. Expected {count_before_merge + 1}, got {_after_merge}"
_t001 = spark.table(TABLE).filter(F.col("customer_id") == "TEST001").collect()
assert len(_t001) == 1, "TEST001 must still exist after MERGE"
assert _t001[0]['first_name']   == 'MergedAlice', \
    f"TEST001 first_name should be 'MergedAlice', got '{_t001[0]['first_name']}'"
assert _t001[0]['loyalty_tier'] == 'Platinum', \
    f"TEST001 loyalty_tier should be 'Platinum', got '{_t001[0]['loyalty_tier']}'"
_new = spark.table(TABLE).filter(F.col("customer_id") == "MERGE_NEW").collect()
assert len(_new) == 1, "MERGE_NEW must be inserted by MERGE"
_hist = [r['operation'] for r in spark.sql(f"DESCRIBE HISTORY {TABLE}").limit(3).collect()]
assert 'MERGE' in _hist, "MERGE must appear in Delta history"
print(f"✓ Task 08 passed — MERGE: 2 updated, 1 inserted | {_after_merge:,} total rows")

---
## Task 09 — Version History

📖 [DESCRIBE HISTORY — Delta transaction log](https://docs.databricks.com/delta/history.html)

In [0]:
full_history   = spark.sql(f"DESCRIBE HISTORY {TABLE}")
version_count  = full_history.count()
operation_list = [r['operation']
                  for r in full_history.orderBy(F.desc("version")).collect()]

print(f"Total versions            : {version_count}")
print(f"Operations (newest first) : {operation_list}")
display(full_history.select("version", "timestamp", "operation").orderBy(F.desc("version")))

In [0]:
# ── CHECK 09 ──────────────────────────────────────────────────────
assert 'full_history'   in dir(), "full_history not defined"
assert 'version_count'  in dir(), "version_count not defined"
assert 'operation_list' in dir(), "operation_list not defined"
assert isinstance(version_count, int) and version_count >= 5, \
    f"Expected ≥5 versions (CREATE+ALTER+ALTER+INSERT+UPDATE+DELETE+MERGE...), got {version_count}"
assert isinstance(operation_list, list) and len(operation_list) == version_count, \
    "operation_list length must equal version_count"
_ops = set(operation_list)
for _op in ['UPDATE', 'DELETE', 'MERGE']:
    assert _op in _ops, f"'{_op}' not found in history — run the previous tasks first"
print(f"✓ Task 09 passed — {version_count} versions: {operation_list}")

---
## Task 10 — Time Travel by version number

📖 [Delta time travel — VERSION AS OF](https://docs.databricks.com/delta/history.html#retrieve-information-by-using-time-travel)

In [0]:
df_v0 = spark.sql(f"SELECT * FROM {TABLE} VERSION AS OF 0")

print(f"Rows at v0      : {df_v0.count():,}")
print(f"Rows in current : {spark.table(TABLE).count():,}")
print(f"Columns at v0   : {df_v0.columns}")

In [0]:
# ── CHECK 10 ──────────────────────────────────────────────────────
assert 'df_v0' in dir(), "df_v0 not defined"
assert 'customer_id' in df_v0.columns, "df_v0 must have customer_id column"
_v0_ids = {r['customer_id'] for r in df_v0.select("customer_id").collect()}
assert 'TEST001' not in _v0_ids, \
    "TEST001 should NOT be in version 0 (it was inserted in Task 05)"
_v0_count      = df_v0.count()
_current_count = spark.table(TABLE).count()
assert _v0_count != _current_count, \
    "Version 0 and current should have different row counts (DML changed the table)"
print(f"✓ Task 10 passed — v0: {_v0_count:,} rows | current: {_current_count:,} rows")

---
## Task 11 — Time Travel by timestamp

📖 [Delta time travel — TIMESTAMP AS OF](https://docs.databricks.com/delta/history.html#retrieve-information-by-using-time-travel)

In [0]:
# Find the INSERT version (WRITE with Append mode) from Task 05
insert_rows = full_history.filter(
    (F.col("operation") == "WRITE") &
    (F.col("operationParameters.mode") == "Append")
).orderBy("version").collect()

ts_insert = insert_rows[0]['timestamp']   # already a Python datetime
ts_str    = (ts_insert + timedelta(seconds=1)).strftime("%Y-%m-%d %H:%M:%S")
df_ts     = spark.sql(f"SELECT * FROM {TABLE} TIMESTAMP AS OF '{ts_str}'")

print(f"Timestamp used : {ts_insert}")
print(f"Rows in df_ts  : {df_ts.count():,}")

In [0]:
# ── CHECK 11 ──────────────────────────────────────────────────────
assert 'ts_insert' in dir(), "ts_insert not defined"
assert 'df_ts'     in dir(), "df_ts not defined"
assert isinstance(ts_insert, datetime), f"ts_insert must be datetime, got {type(ts_insert)}"
_ts_ids = {r['customer_id'] for r in df_ts.select("customer_id").collect()}
assert 'TEST001' in _ts_ids, "df_ts must contain TEST001 (inserted in Task 05)"
_row = df_ts.filter(F.col("customer_id") == "TEST001").collect()[0]
assert _row['country'] == 'TestLand', \
    f"At INSERT time, TEST001.country should be 'TestLand' (before UPDATE), got '{_row['country']}'"
print(f"✓ Task 11 passed — timestamp travel OK | TEST001.country='TestLand' at insert time")

---
## Task 12 ⭐ — RESTORE — Disaster Recovery

📖 [RESTORE TABLE — Delta Lake](https://docs.databricks.com/sql/language-manual/delta-restore.html)

In [0]:
rows_before_disaster = spark.table(TABLE).count()

# Step 1 — record the current latest version (safe state)
safe_version = (
    spark.sql(f"DESCRIBE HISTORY {TABLE}")
         .orderBy(F.desc("version"))
         .collect()[0]['version']
)

# Step 2 — simulate the disaster
spark.sql(f"DELETE FROM {TABLE}")
print(f"Rows after disaster : {spark.table(TABLE).count():,}  ← should be 0")

# Step 3 — restore
spark.sql(f"RESTORE TABLE {TABLE} TO VERSION AS OF {safe_version}")

rows_after_restore = spark.table(TABLE).count()
print(f"Before disaster : {rows_before_disaster:,}")
print(f"After restore   : {rows_after_restore:,}")

In [0]:
# ── CHECK 12 ──────────────────────────────────────────────────────
_restored = spark.table(TABLE).count()
assert _restored == rows_before_disaster, \
    f"After RESTORE, expected {rows_before_disaster:,} rows but got {_restored:,}"
assert _restored > 0, "Table must not be empty after restore"
_hist_ops = [r['operation'] for r in spark.sql(f"DESCRIBE HISTORY {TABLE}").limit(5).collect()]
assert 'RESTORE' in _hist_ops, "RESTORE must appear in Delta history"
print(f"✓ Task 12 passed — disaster recovered! {_restored:,} rows back, RESTORE in history")

---
## 🎉 Workshop Complete!

| Task | Topic | Key Command |
|------|-------|-------------|
| 01 | Create Delta table | `write.format("delta").saveAsTable` |
| 02 | Schema Enforcement | Delta rejects writes with wrong schema |
| 03 | Schema Evolution | `ALTER TABLE ADD COLUMN` |
| 04 | Constraints | `ALTER TABLE ADD CONSTRAINT ... CHECK (...)` |
| 05 | INSERT | `INSERT INTO ... VALUES` |
| 06 | UPDATE | `UPDATE ... SET ... WHERE` |
| 07 | DELETE | `DELETE FROM ... WHERE ... IN` |
| 08 | MERGE INTO | `MERGE INTO ... USING ... ON ... WHEN MATCHED ...` |
| 09 | Version History | `DESCRIBE HISTORY` |
| 10 | Time Travel — version | `VERSION AS OF 0` |
| 11 | Time Travel — timestamp | `TIMESTAMP AS OF 'ts'` |
| 12 ⭐ | Disaster Recovery | `RESTORE TABLE ... TO VERSION AS OF N` |

**Next:** M05 — Orchestration & Lakeflow Pipelines